# Polysemantic tokens: max-pooled layers, no neuron-position table

The 29 saved Qwen2.5-7B answer-token layers are **max pooled** over 3-layer, stride-2 windows (29 → 15), giving a 15 × 3,584 matrix. Each hidden unit is one polysemantic token: its 15-value trajectory, standardized with source-train statistics, through Linear(15 → 128). **No learned per-neuron position table**, so the model is hidden-size agnostic.

Two runs, both with the Perceiver encoder (64 learned 512-d latents cross-attend to the 3,584 neuron tokens, 2 latent self-attention blocks, mean over latents = 512-d embedding → residual Mish 4 × 128 MLP head) and AdamW lr 3e-4 with 5% warmup + cosine:
* **descriptors**: each neuron's source-train mean, spread and label correlation per pooled layer → MLP → added to its token (neuron identity from behaviour, not index)
* **pure set**: no descriptors either — the model sees an unordered set of neuron trajectories

Same six-source split, BCE, batch 128, weight decay 1e-4, seed 42, 10 epochs, best-source-validation-AUROC checkpoint, threshold calibration and held-out TriviaQA.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 627551a
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
BASE = [sys.executable, '/kaggle/working/SALA/sala_polysemantic.py', '--pooling', 'max1d', '--encoder', 'perceiver', '--neuron-positions', 'none', '--latents', '64', '--latent-heads', '8', '--model-dim', '128', '--embedding-dim', '512', '--blocks', '2', '--hidden-dim', '128', '--hidden-layers', '4', '--learning-rate', '3e-4', '--lr-schedule', 'cosine']
RUNS = {'maxpool_descriptors': BASE, 'maxpool_pure_set': BASE + ['--no-descriptors']}
OUTPUT_DIRS = {}
for name, base in RUNS.items():
    output_dir = Path(f'/kaggle/working/sala_g14_6src_polysemantic_{name}')
    OUTPUT_DIRS[name] = output_dir
    command = base + ['--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(output_dir), '--bleurt-threshold', '0.5', '--epochs-erm', '10', '--seed', '42', '--train-device', 'cuda:0']
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
rows = []
for name, output_dir in OUTPUT_DIRS.items():
    s = json.loads((output_dir / 'summary.json').read_text())
    rows.append({'model': name, 'parameters': s.get('parameters'), 'best_epoch': s['best_epoch'],
                 'source_val_auroc': s['best_source_validation_auroc_during_training'],
                 'source_test_auroc': s['source_test_metrics']['auroc'],
                 'target_auroc': s['target_metrics']['auroc'],
                 'target_balanced_accuracy': s['target_metrics']['balanced_accuracy'],
                 'target_f1': s['target_metrics']['f1']})
display(pd.DataFrame(rows))
print(pd.DataFrame(rows).to_string())